In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 · Gold Layer
# MAGIC Builds a small reporting-ready star schema on top of Silver's
# MAGIC **current** rows (`is_current = true`). This is what Power BI /
# MAGIC downstream analytics connects to — never Bronze/Silver directly.

# COMMAND ----------
import sys, os
sys.path.append(os.path.abspath("../common"))
from config import silver_table, gold_table

# COMMAND ----------
# MAGIC %md ### dim_patient

# COMMAND ----------
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_table('dim_patient')} AS
SELECT
    resource_id AS patient_id,
    given_name, family_name,
    concat_ws(' ', given_name, family_name) AS full_name,
    gender, birth_date,
    date_diff(current_date(), to_date(birth_date)) / 365.25 AS age_years,
    city, state, country,
    effective_start_date, last_updated
FROM {silver_table('Patient')}
WHERE is_current = true
""")

# COMMAND ----------
# MAGIC %md ### fact_encounter

# COMMAND ----------
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_table('fact_encounter')} AS
SELECT
    e.resource_id AS encounter_id,
    e.patient_id,
    e.status, e.class_code,
    to_timestamp(e.period_start) AS period_start,
    to_timestamp(e.period_end)   AS period_end,
    (unix_timestamp(e.period_end) - unix_timestamp(e.period_start)) / 60.0 AS duration_minutes
FROM {silver_table('Encounter')} e
WHERE e.is_current = true
""")

# COMMAND ----------
# MAGIC %md ### fact_observation

# COMMAND ----------
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_table('fact_observation')} AS
SELECT
    o.resource_id AS observation_id,
    o.patient_id, o.encounter_id,
    o.code, o.code_display,
    o.value_quantity, o.value_unit,
    to_timestamp(o.effective_datetime) AS effective_datetime,
    o.status
FROM {silver_table('Observation')} o
WHERE o.is_current = true
""")

# COMMAND ----------
# MAGIC %md ### fact_condition

# COMMAND ----------
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_table('fact_condition')} AS
SELECT
    c.resource_id AS condition_id,
    c.patient_id, c.encounter_id,
    c.clinical_status, c.code, c.code_display,
    to_date(c.onset_datetime) AS onset_date,
    to_date(c.recorded_date)  AS recorded_date
FROM {silver_table('Condition')} c
WHERE c.is_current = true
""")

# COMMAND ----------
# MAGIC %md ### patient_360 — denormalized view for the optional Power BI extension

# COMMAND ----------
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_table('patient_360')} AS
SELECT
    p.patient_id, p.full_name, p.gender, p.age_years, p.city, p.state, p.country,
    count(DISTINCT e.encounter_id) AS total_encounters,
    count(DISTINCT o.observation_id) AS total_observations,
    count(DISTINCT c.condition_id) AS total_conditions,
    max(e.period_start) AS last_encounter_date
FROM {gold_table('dim_patient')} p
LEFT JOIN {gold_table('fact_encounter')} e ON p.patient_id = e.patient_id
LEFT JOIN {gold_table('fact_observation')} o ON p.patient_id = o.patient_id
LEFT JOIN {gold_table('fact_condition')} c ON p.patient_id = c.patient_id
GROUP BY p.patient_id, p.full_name, p.gender, p.age_years, p.city, p.state, p.country
""")

print("Gold layer built: dim_patient, fact_encounter, fact_observation, "
      "fact_condition, patient_360.")
